In [3]:
import collections

# --- Configuration & Data Mapping ---
ITEM_NAMES = {0:"circle", 1:"pentagon", 2:"trapezoid", 3:"triangle", 4:"star", 5:"moon", 6:"heart"}

# 11 Orders assigned to conveyors
ORDERS_CONFIG = [
    {"id": 1,  "belt": 1, "reqs": {3: 3, 1: 2}}, 
    {"id": 2,  "belt": 2, "reqs": {2: 1, 3: 3, 4: 1}}, 
    {"id": 3,  "belt": 3, "reqs": {5: 2}}, 
    {"id": 4,  "belt": 4, "reqs": {0: 3, 5: 1}}, 
    {"id": 5,  "belt": 1, "reqs": {1: 1, 2: 1}}, 
    {"id": 6,  "belt": 2, "reqs": {1: 2}}, 
    {"id": 7,  "belt": 1, "reqs": {5: 1, 3: 2}}, 
    {"id": 8,  "belt": 2, "reqs": {4: 2, 2: 1}}, 
    {"id": 9,  "belt": 3, "reqs": {5: 1, 0: 3, 1: 2}}, 
    {"id": 10, "belt": 4, "reqs": {6: 3}}, 
    {"id": 11, "belt": 1, "reqs": {1: 1}}, 
]

# Tote Contents based on the provided data matrix
TOTE_CONTENTS = {
    0:  [(3, 3), (1, 2)], 14: [(2, 1), (4, 1)], 4:  [(3, 3)],
    7:  [(5, 2)], 9:  [(0, 3)], 10: [(5, 1), (1, 1)],
    11: [(1, 1)], 12: [(2, 1)], 1:  [(1, 2), (6, 3)],
    3:  [(5, 1)], 13: [(3, 2), (5, 1), (1, 2)], 8:  [(4, 2), (2, 1), (0, 3)]
}

TOTE_SEQUENCE = [0, 14, 4, 7, 9, 10, 11, 12, 1, 3, 13, 8]

# Build FIFO Induction Queue
induction_queue = []
for t_id in TOTE_SEQUENCE:
    for item_type, qty in TOTE_CONTENTS[t_id]:
        for _ in range(qty): induction_queue.append(item_type)

class RobustConveyorSim:
    def __init__(self, orders, queue):
        self.time = 0
        self.queue = queue.copy()
        self.completed_count = 0
        self.total_orders = len(orders)
        
        # [Slot 0: Middle/Scanner, Slot 1: End/Transfer]
        self.belts = {1: [None, None], 2: [None, None], 3: [None, None], 4: [None, None]}
        self.schedules = {1: [], 2: [], 3: [], 4: []}
        for o in orders: self.schedules[o["belt"]].append(o)
        self.active_orders = {b: self.schedules[b].pop(0) if self.schedules[b] else None for b in range(1, 5)}
        self.log = []

    def get_items_in_loop(self):
        return sum(1 for b in self.belts.values() for slot in b if slot is not None)

    def step(self):
        self.time += 1
        
        # 1. Inter-Belt Transfer (End of N to Middle of N+1)
        next_map = {1: 2, 2: 3, 3: 4, 4: 1}
        for curr_b in [4, 3, 2, 1]:
            nxt_b = next_map[curr_b]
            if self.belts[curr_b][1] is not None and self.belts[nxt_b][0] is None:
                item = self.belts[curr_b][1]
                self.belts[nxt_b][0] = item
                self.belts[curr_b][1] = None
                if nxt_b == 1: print(f"T={self.time:03d}s | Loop: {ITEM_NAMES[item]} recirculated to B1")

        # 2. Diversion Logic
        for b_id in range(1, 5):
            item = self.belts[b_id][0]
            if item is not None:
                order = self.active_orders[b_id]
                if order and item in order["reqs"] and order["reqs"][item] > 0:
                    order["reqs"][item] -= 1
                    self.belts[b_id][0] = None
                    print(f"T={self.time:03d}s | Belt {b_id}: Diverted {ITEM_NAMES[item]} for Order {order['id']}")
                    
                    if sum(order["reqs"].values()) == 0:
                        self.completed_count += 1
                        self.log.append((order['id'], b_id, self.time))
                        print(f"T={self.time:03d}s | *** Order {order['id']} COMPLETE on Belt {b_id} ***")
                        self.active_orders[b_id] = self.schedules[b_id].pop(0) if self.schedules[b_id] else None

        # 3. Internal Movement (Middle to End)
        for b_id in range(1, 5):
            if self.belts[b_id][0] is not None and self.belts[b_id][1] is None:
                self.belts[b_id][1] = self.belts[b_id][0]
                self.belts[b_id][0] = None

        # 4. Controlled Induction
        # Logic: Don't induct if the loop is at 87% capacity (7/8 slots) to prevent deadlock
        if self.queue and self.belts[1][0] is None and self.get_items_in_loop() < 7:
            new_item = self.queue.pop(0)
            self.belts[1][0] = new_item
            print(f"T={self.time:03d}s | System: Inducted {ITEM_NAMES[new_item]} (Queue size: {len(self.queue)})")

    def run(self):
        print("Starting Benchmark Simulation...")
        while self.completed_count < self.total_orders and self.time < 5000:
            self.step()
        
        print("\n" + "="*50)
        print(f"{'Order ID':<10} | {'Belt Used':<12} | {'Time to Complete'}")
        print("-" * 50)
        for oid, b, t in self.log:
            print(f"Order {oid:<6} | Belt {b:<9} | {t}s")
        print("="*50)
        print(f"TOTAL CUMULATIVE TIME: {self.time} seconds")
        print("="*50)

# Run Simulation
sim = RobustConveyorSim(ORDERS_CONFIG, induction_queue)
sim.run()

Starting Benchmark Simulation...
T=001s | System: Inducted triangle (Queue size: 35)
T=002s | Belt 1: Diverted triangle for Order 1
T=002s | System: Inducted triangle (Queue size: 34)
T=003s | Belt 1: Diverted triangle for Order 1
T=003s | System: Inducted triangle (Queue size: 33)
T=004s | Belt 1: Diverted triangle for Order 1
T=004s | System: Inducted pentagon (Queue size: 32)
T=005s | Belt 1: Diverted pentagon for Order 1
T=005s | System: Inducted pentagon (Queue size: 31)
T=006s | Belt 1: Diverted pentagon for Order 1
T=006s | *** Order 1 COMPLETE on Belt 1 ***
T=006s | System: Inducted trapezoid (Queue size: 30)
T=007s | Belt 1: Diverted trapezoid for Order 5
T=007s | System: Inducted star (Queue size: 29)
T=008s | System: Inducted triangle (Queue size: 28)
T=009s | Belt 2: Diverted star for Order 2
T=009s | System: Inducted triangle (Queue size: 27)
T=010s | Belt 2: Diverted triangle for Order 2
T=010s | System: Inducted triangle (Queue size: 26)
T=011s | Belt 2: Diverted triangl